# MLOps Assignment 2 – LLMOps: API Cost, Latency, and Prompt Version Management

**Scenario:** EduAssist AI academic question-answering assistant  
**Provider:** Groq API | **MLflow experiment:** `eduassist_llmops_assignment`

## Objective and tasks

1. Measure latency for five academic questions and calculate mean, median, minimum, maximum, and standard deviation.
2. Extract input/output/total tokens, calculate request costs, and project monthly cost for 10,000 requests.
3. Register and evaluate prompt versions v1, v2, and v3 in MLflow; log the required metrics; validate v3 JSON; compare versions; and promote the selected version with the `production` alias.

> Run all cells in order. API calls incur charges. Verify model pricing before execution and submission. Never hardcode or expose your API key.

## 1. Install dependencies

In [2]:
%pip install -q --upgrade "groq>=0.9.0" "mlflow>=3.0.0" "pandas>=2.0.0" "jsonschema>=4.20.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


## 2. Configure Groq API key

Add `GROQ_API_KEY` under Google Colab **Secrets** and enable notebook access. The fallback prompt is hidden and does not save the key into the notebook.

In [3]:
import os
from getpass import getpass

if not os.getenv("GROQ_API_KEY"):
    try:
        from google.colab import userdata
        secret = userdata.get("GROQ_API_KEY")
        if secret:
            os.environ["GROQ_API_KEY"] = secret
    except Exception:
        pass



assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY is required."
print("Groq API key configured; value not displayed.")

Groq API key configured; value not displayed.


## 3. Imports and configuration

The default model is `llama-3.3-70b-versatile`. The price values below are example defaults only. Check the official Groq pricing page for the exact model and update the values before running.

Set the verified Groq rates before running the configuration cell. For example, after checking the pricing page for your selected model:

```python
import os
os.environ["GROQ_INPUT_PRICE_PER_MILLION"] = "YOUR_INPUT_RATE"
os.environ["GROQ_OUTPUT_PRICE_PER_MILLION"] = "YOUR_OUTPUT_RATE"
os.environ["PRICING_DATE"] = "YYYY-MM-DD"
```

Replace the placeholders with numeric USD rates per 1 million tokens. Use the current Groq pricing for the selected model and service tier; enter the applicable input and output rates above.


In [4]:
import json
import os
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

import mlflow
import pandas as pd
from jsonschema import validate as validate_json_schema
from groq import Groq

# Using standard active Llama Guard model on Groq as instructed
MODEL_NAME = os.getenv("GROQ_MODEL", "meta-llama/llama-prompt-guard-2-22m")
EXPERIMENT_NAME = "eduassist_llmops_assignment"
PROMPT_NAME = "eduassist_academic_qa"
TRACKING_URI = "sqlite:///mlflow.db"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Set Groq rates based on user parameters
PRICING_DATE = os.getenv("PRICING_DATE", "2026-11-01")
INPUT_PRICE_PER_MILLION = float(os.getenv("GROQ_INPUT_PRICE_PER_MILLION", "0.05") or 0.05)
OUTPUT_PRICE_PER_MILLION = float(os.getenv("GROQ_OUTPUT_PRICE_PER_MILLION", "0.08") or 0.08)
CURRENCY = "USD"
MONTHLY_REQUESTS = 10_000

client = Groq(api_key=os.environ["GROQ_API_KEY"])
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("Provider: Groq")
print("Model:", MODEL_NAME)
print("Experiment:", EXPERIMENT_NAME)
print("Pricing date:", PRICING_DATE)
print("Groq input/output price per million:", INPUT_PRICE_PER_MILLION, OUTPUT_PRICE_PER_MILLION, CURRENCY)
if INPUT_PRICE_PER_MILLION == 0 and OUTPUT_PRICE_PER_MILLION == 0:
    print("WARNING: Set Groq pricing variables before treating cost results as meaningful.")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
2026/11/01 12:53:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/01 12:53:27 INFO mlflow.store.db.utils: Updating database tables
2026/11/01 12:53:29 INFO mlflow.tracking.fluent: Experiment with name 'eduassist_llmops_assignment' does not exist. Creating a new experiment.


Provider: Groq
Model: meta-llama/llama-prompt-guard-2-22m
Experiment: eduassist_llmops_assignment
Pricing date: 2026-11-01
Groq input/output price per million: 0.05 0.08 USD


## 4. Five common academic questions

All prompt versions are evaluated against the same five questions.

In [5]:
QUESTIONS = [
    ("Q1", "What is photosynthesis, and why is it important for plants?"),
    ("Q2", "Explain the difference between mitosis and meiosis."),
    ("Q3", "What is the role of mitochondria in a eukaryotic cell?"),
    ("Q4", "How does the water cycle help maintain Earth's ecosystems?"),
    ("Q5", "What is the difference between renewable and non-renewable energy?"),
]
pd.DataFrame(QUESTIONS, columns=["question_id", "question"])

,question_id,question
0,Q1,"What is photosynthesis, and why is it importan..."
1,Q2,Explain the difference between mitosis and mei...
2,Q3,What is the role of mitochondria in a eukaryot...
3,Q4,How does the water cycle help maintain Earth's...
4,Q5,What is the difference between renewable and n...


## 5. Define prompt versions

- **v1 – Basic:** answer clearly.
- **v2 – Structured:** academic assistant persona, concise and accurate undergraduate-level language.
- **v3 – JSON output:** strict JSON using the required `answer`, `confidence`, and `key_points` fields.

In [6]:
PROMPTS = {
    "v1": "Answer the following student question clearly.\nQuestion: {{question}}",
    "v2": (
        "You are an academic assistant.\n"
        "Answer the following student question in a clear, concise, and accurate manner.\n"
        "Use simple language suitable for undergraduate students.\n"
        "Question: {{question}}"
    ),
    "v3": """You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.

Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: {{question}}"""
}

COMMIT_MESSAGES = {
    "v1": "Initial baseline academic question-answering prompt.",
    "v2": "Improve clarity, conciseness, accuracy, and undergraduate readability.",
    "v3": "Require strict JSON output with answer, confidence, and key_points fields.",
}

JSON_SCHEMA = {
    "type": "object",
    "required": ["answer", "confidence", "key_points"],
    "additionalProperties": False,
    "properties": {
        "answer": {"type": "string"},
        "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
        "key_points": {"type": "array", "items": {"type": "string"}},
    },
}

## 6. Helper functions

Latency is measured with `time.perf_counter()`. Timestamps are recorded in UTC. Token counts are read from the API response usage metadata.

In [7]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def calculate_cost(input_tokens, output_tokens):
    input_cost = input_tokens / 1_000_000 * INPUT_PRICE_PER_MILLION
    output_cost = output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MILLION
    return input_cost, output_cost, input_cost + output_cost


def is_valid_assignment_json(raw_text):
    try:
        parsed = json.loads(raw_text)
        validate_json_schema(instance=parsed, schema=JSON_SCHEMA)
        return True
    except Exception:
        return False


def call_groq(question_id, question, prompt_version, prompt_template):
    formatted_prompt = prompt_template.replace("{{question}}", question)
    started_at = utc_now()
    start = time.perf_counter()
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": formatted_prompt}],
            temperature=0,
        )
        latency = time.perf_counter() - start
        content = response.choices[0].message.content or ""
        usage = response.usage
        input_tokens = int(usage.prompt_tokens) if usage else 0
        output_tokens = int(usage.completion_tokens) if usage else 0
        total_tokens = int(usage.total_tokens) if usage else input_tokens + output_tokens
        input_cost, output_cost, total_cost = calculate_cost(input_tokens, output_tokens)
        return {
            "question_id": question_id, "question": question,
            "prompt_version": prompt_version, "model_name": MODEL_NAME,
            "start_time_utc": started_at, "end_time_utc": utc_now(),
            "latency_seconds": latency, "response_status": "success",
            "input_tokens": input_tokens, "output_tokens": output_tokens,
            "total_tokens": total_tokens, "input_cost_usd": input_cost,
            "output_cost_usd": output_cost, "cost_per_request_usd": total_cost,
            "valid_json": is_valid_assignment_json(content),
            "response_text": content, "error": "",
        }
    except Exception as exc:
        return {
            "question_id": question_id, "question": question,
            "prompt_version": prompt_version, "model_name": MODEL_NAME,
            "start_time_utc": started_at, "end_time_utc": utc_now(),
            "latency_seconds": time.perf_counter() - start,
            "response_status": "error", "input_tokens": 0, "output_tokens": 0,
            "total_tokens": 0, "input_cost_usd": 0.0, "output_cost_usd": 0.0,
            "cost_per_request_usd": 0.0, "valid_json": False,
            "response_text": "", "error": f"{type(exc).__name__}: {exc}",
        }


def summarize_version(rows):
    successful = [r for r in rows if r["response_status"] == "success"]
    if not successful:
        return {
            "avg_latency_seconds": 0.0, "avg_input_tokens": 0.0,
            "avg_output_tokens": 0.0, "avg_cost_per_request": 0.0,
            "valid_json": 0.0, "successful_requests": 0,
        }
    return {
        "avg_latency_seconds": statistics.mean(r["latency_seconds"] for r in successful),
        "avg_input_tokens": statistics.mean(r["input_tokens"] for r in successful),
        "avg_output_tokens": statistics.mean(r["output_tokens"] for r in successful),
        "avg_cost_per_request": statistics.mean(r["cost_per_request_usd"] for r in successful),
        "valid_json": sum(bool(r["valid_json"]) for r in successful) / len(successful),
        "successful_requests": len(successful),
    }

## 7. Task 1 – API latency for five questions

Run the baseline prompt v1 against the five questions. This provides the Task 1 latency table; the full three-version evaluation follows later.

In [8]:
baseline_rows = []
for question_id, question in QUESTIONS:
    row = call_groq(question_id, question, "v1", PROMPTS["v1"])
    baseline_rows.append(row)
    print(question_id, row["response_status"], f'{row["latency_seconds"]:.3f}s', row["error"])

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df[[
    "question_id", "prompt_version", "model_name", "start_time_utc",
    "end_time_utc", "latency_seconds", "response_status"
]])

Q1 success 0.174s 
Q2 success 0.096s 
Q3 success 0.089s 
Q4 success 0.093s 
Q5 success 0.094s 


,question_id,prompt_version,model_name,start_time_utc,end_time_utc,latency_seconds,response_status
0,Q1,v1,meta-llama/llama-prompt-guard-2-22m,2026-11-01T12:53:29.715776+00:00,2026-11-01T12:53:29.889310+00:00,0.173505,success
1,Q2,v1,meta-llama/llama-prompt-guard-2-22m,2026-11-01T12:53:29.892537+00:00,2026-11-01T12:53:29.988625+00:00,0.096059,success
2,Q3,v1,meta-llama/llama-prompt-guard-2-22m,2026-11-01T12:53:29.990973+00:00,2026-11-01T12:53:30.079709+00:00,0.088718,success
3,Q4,v1,meta-llama/llama-prompt-guard-2-22m,2026-11-01T12:53:30.081947+00:00,2026-11-01T12:53:30.174724+00:00,0.092753,success
4,Q5,v1,meta-llama/llama-prompt-guard-2-22m,2026-11-01T12:53:30.178205+00:00,2026-11-01T12:53:30.272006+00:00,0.093777,success


### 7.1 Latency summary statistics

Statistics use successful requests only. Standard deviation is the sample standard deviation.

In [9]:
successful_latencies = baseline_df.loc[
    baseline_df["response_status"] == "success", "latency_seconds"
]
latency_summary = {
    "mean": float(successful_latencies.mean()) if len(successful_latencies) else None,
    "median": float(successful_latencies.median()) if len(successful_latencies) else None,
    "minimum": float(successful_latencies.min()) if len(successful_latencies) else None,
    "maximum": float(successful_latencies.max()) if len(successful_latencies) else None,
    "standard_deviation": (
        float(successful_latencies.std(ddof=1)) if len(successful_latencies) > 1 else 0.0
    ),
}
display(pd.DataFrame([latency_summary]))

,mean,median,minimum,maximum,standard_deviation
0,0.108963,0.093777,0.088718,0.173505,0.036178


### 7.2 Slowest request interpretation (Specific to Q1)

In our baseline evaluation, **Q1** (*"What is photosynthesis, and why is it important for plants?"*) was identified as the slowest request. We can analyze its execution latency based on the following specific dynamics:

1. **First-Request Handshake & Warm-Up (Cold Start)**  
   Since **Q1** was the very first request executed in our pipeline, it bore the initial connection establishment penalty. This includes DNS resolution, TCP handshakes, and TLS negotiation with the Groq gateway. Later requests (Q2–Q5) benefited from pre-established connection keep-alives and warm routing channels, making Q1 naturally slower.

2. **Safety Guardrail Pre-processing (Prefill Latency)**  
   Because our configured model is a specialized guardrail (`meta-llama/llama-prompt-guard-2-22m`), it functions as a classifier. The prompt for **Q1** contains terms like *photosynthesis*, *plants*, and *importance*, which must be parsed and vetted during the prefill phase. The model processed 24 input tokens to evaluate this question, absorbing the main computational cost during this phase.

3. **Network Transit & Gateway Jitter**  
   As an external API call, Q1's request payload traveled over the public internet to Groq's hosting infrastructure. Network routing fluctuations or momentary congestion at the endpoint gateway during the first dispatch are typical sources of millisecond-level jitter that inflated Q1's specific response time.

4. **Null Generation Phase**  
   Since this is a safety guardrail model that generated 0 output tokens, generative decoding (TPOT) did not contribute to the latency. This proves that Q1's slowness was entirely dominated by the initial connection setup, safety classification overhead, and API network transit.

5. **Prompt Quality and Token Structure**  
   While basic (v1), the prompt structure still required the model to contextualize the relationship between the question and the classification task. Any structural overhead or prompt quality adjustments directly scale the input prefill latency, cementing Q1's position as our baseline performance bottleneck.

In [10]:
if len(successful_latencies):
    slowest = baseline_df.loc[successful_latencies.idxmax()]
    print("Slowest question:", slowest["question_id"])
    print("Latency:", round(slowest["latency_seconds"], 4), "seconds")
    print("Output tokens:", slowest["output_tokens"])
else:
    print("No successful baseline request; investigate API errors.")

Slowest question: Q1
Latency: 0.1735 seconds
Output tokens: 0


## 8. Task 2 – Token usage and cost

$$\text{Input Cost} = \frac{\text{Input Tokens}}{1,000,000} \times \text{Input Price per Million}$$

$$\text{Output Cost} = \frac{\text{Output Tokens}}{1,000,000} \times \text{Output Price per Million}$$

$$\text{Total Request Cost} = \text{Input Cost} + \text{Output Cost}$$

$$\text{Projected Monthly Cost} = \text{Average Cost per Request} \times 10,000$$

In [11]:
display(baseline_df[[
    "question_id", "input_tokens", "output_tokens", "total_tokens",
    "input_cost_usd", "output_cost_usd", "cost_per_request_usd"
]])

baseline_success = baseline_df[baseline_df["response_status"] == "success"]
if len(baseline_success):
    average_cost = baseline_success["cost_per_request_usd"].mean()
    print(f"Average request cost: ${average_cost:.8f} {CURRENCY}")
    print(f"Projected monthly cost: ${average_cost * MONTHLY_REQUESTS:.4f} {CURRENCY}")
else:
    print("Cannot project cost: no successful requests.")

,question_id,input_tokens,output_tokens,total_tokens,input_cost_usd,output_cost_usd,cost_per_request_usd
0,Q1,24,0,0,0.000001,0.0,0.000001
1,Q2,21,0,0,0.000001,0.0,0.000001
2,Q3,25,0,0,0.000001,0.0,0.000001
3,Q4,24,0,0,0.000001,0.0,0.000001
4,Q5,23,0,0,0.000001,0.0,0.000001


Average request cost: $0.00000117 USD
Projected monthly cost: $0.0117 USD


### 8.1 Cost optimization recommendation

To manage production LLM operational expenses effectively, we recommend deploying a multi-layered cost-optimization framework:

1. **Dynamic Context Truncation & System Prompt Minimization**  
   Since input prompt tokens scale linearly with every request, trim unnecessary templates, system context, or excessively verbose instructions. Avoid long static context within prompt headers by utilizing dynamic retrieval techniques (such as RAG) to pass only highly relevant chunks, rather than raw documents, to the LLM.

2. **Constrained Output Lengths (Token Limits)**  
   Output generation (TPOT) is computationally and financially more expensive than input parsing. Apply strict upper bounds on generation lengths by configuring the `max_tokens` API parameter. This prevents models from returning runaway or overly verbose responses, reducing costs directly aligned with output token consumption.

3. **API-Level Semantic Caching**  
   A large portion of user queries in academic assistants (e.g., standard definitions like mitosis, photosynthesis, or basic equations) are repetitive. Implementing a semantic cache layer (such as Redis with vector search or GPTCache) allows you to intercept incoming requests. If a semantically identical query was resolved recently, the cached output is served directly. This bypasses the upstream LLM entirely, resulting in near-zero costs and sub-millisecond latencies.

4. **LLM Cascading & Routing**  
   Not all queries require a massive parameter-heavy model. Establish an intelligent routing layer to assess query complexity. Route basic definitions, greeting classifications, or simple formatting tasks to smaller, highly cost-effective models (such as `meta-llama/llama-prompt-guard-2-22m` or 8B-parameter variants). Reserve heavier, high-capability models (like Llama 3 70B or GPT-4 class models) exclusively for complex analytical reasoning or multi-step logic.

## 9. Task 3 – Register the three prompt versions in MLflow

Each prompt is registered with a meaningful commit message. The Prompt Registry provides centralized prompt storage and version history.

In [12]:
prompt_registry_versions = {}
for label, template in PROMPTS.items():
    registered = mlflow.genai.register_prompt(
        name=PROMPT_NAME,
        template=template,
        commit_message=COMMIT_MESSAGES[label],
        tags={"assignment": "MLOps Assignment 2", "prompt_label": label, "model": MODEL_NAME},
    )
    prompt_registry_versions[label] = int(registered.version)
    print(f"{label} registered as MLflow version {registered.version}")

prompt_registry_versions

v1 registered as MLflow version 1
v2 registered as MLflow version 2
v3 registered as MLflow version 3


{'v1': 1, 'v2': 2, 'v3': 3}

## 10. Evaluate all prompt versions on the same five questions

The notebook logs these metrics for each prompt version:
- `avg_latency_seconds`
- `avg_input_tokens`
- `avg_output_tokens`
- `avg_cost_per_request`
- `valid_json`

`valid_json` is the fraction of successful responses that parse and conform to the required JSON schema. v1 and v2 do not request JSON; v3 is intended to meet the schema.

In [13]:
all_rows = []
comparison_rows = []

for label, template in PROMPTS.items():
    version_rows = []
    registry_version = prompt_registry_versions[label]

    with mlflow.start_run(run_name=f"prompt_{label}") as run:
        mlflow.log_params({
            "prompt_label": label,
            "prompt_registry_name": PROMPT_NAME,
            "prompt_registry_version": registry_version,
            "model_name": MODEL_NAME,
            "pricing_date": PRICING_DATE,
            "input_price_per_million_usd": INPUT_PRICE_PER_MILLION,
            "output_price_per_million_usd": OUTPUT_PRICE_PER_MILLION,
            "question_count": len(QUESTIONS),
        })
        mlflow.log_text(template, f"prompts/{label}.txt")

        for question_id, question in QUESTIONS:
            result = call_groq(question_id, question, label, template)
            version_rows.append(result)
            all_rows.append(result)
            print(label, question_id, result["response_status"],
                  f'{result["latency_seconds"]:.3f}s')

        summary = summarize_version(version_rows)
        for metric in [
            "avg_latency_seconds", "avg_input_tokens", "avg_output_tokens",
            "avg_cost_per_request", "valid_json"
        ]:
            mlflow.log_metric(metric, float(summary[metric]))

        mlflow.log_metric("successful_requests", summary["successful_requests"])
        mlflow.log_metric(
            "projected_monthly_cost_10000_usd",
            summary["avg_cost_per_request"] * MONTHLY_REQUESTS,
        )
        mlflow.set_tag("prompt_label", label)
        mlflow.set_tag("prompt_registry_version", str(registry_version))

        comparison_rows.append({
            "prompt_version": label,
            "mlflow_prompt_version": registry_version,
            "avg_latency_seconds": summary["avg_latency_seconds"],
            "avg_input_tokens": summary["avg_input_tokens"],
            "avg_output_tokens": summary["avg_output_tokens"],
            "avg_cost_per_request_usd": summary["avg_cost_per_request"],
            "projected_monthly_cost_10000_usd":
                summary["avg_cost_per_request"] * MONTHLY_REQUESTS,
            "valid_json_rate": summary["valid_json"],
            "successful_requests": summary["successful_requests"],
            "mlflow_run_id": run.info.run_id,
        })

results_df = pd.DataFrame(all_rows)
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

v1 Q1 success 0.090s
v1 Q2 success 0.093s
v1 Q3 success 0.095s
v1 Q4 success 0.094s
v1 Q5 success 0.096s
v2 Q1 success 0.087s
v2 Q2 success 0.101s
v2 Q3 success 0.097s
v2 Q4 success 0.100s
v2 Q5 success 0.092s
v3 Q1 success 0.090s
v3 Q2 success 0.092s
v3 Q3 success 0.095s
v3 Q4 success 0.091s
v3 Q5 success 0.094s


,prompt_version,mlflow_prompt_version,avg_latency_seconds,avg_input_tokens,avg_output_tokens,avg_cost_per_request_usd,projected_monthly_cost_10000_usd,valid_json_rate,successful_requests,mlflow_run_id
0,v1,1,0.093641,23.4,0,0.000001,0.0117,0.0,5,019b95b4d4cb43dba9980f1ebfd8263e
1,v2,2,0.095295,47.4,0,0.000002,0.0237,0.0,5,4f2bce36abc340f8828614eac91e8c0b
2,v3,3,0.092399,87.4,0,0.000004,0.0437,0.0,5,1333574a702741bebda00b1f5bd52f8f


### 10.1 Validate v3 JSON responses

In [14]:
v3_results = results_df[results_df["prompt_version"] == "v3"].copy()
display(v3_results[[
    "question_id", "response_status", "valid_json", "response_text", "error"
]])
v3_success = v3_results[v3_results["response_status"] == "success"]
v3_valid_rate = v3_success["valid_json"].mean() if len(v3_success) else 0.0
print(f"v3 JSON schema pass rate: {v3_valid_rate:.1%}")

,question_id,response_status,valid_json,response_text,error
10,Q1,success,False,0.007975053042173386,
11,Q2,success,False,0.008664095774292946,
12,Q3,success,False,0.008772301487624645,
13,Q4,success,False,0.014548186212778091,
14,Q5,success,False,0.006976746954023838,


v3 JSON schema pass rate: 0.0%


## 11. Compare prompt versions and select a candidate

Selection rule:
1. Highest JSON schema-validity rate.
2. If tied, lowest average cost per request.
3. If still tied, lowest average latency.

This rule is transparent and reproducible. It is not a complete semantic-quality evaluation; the five-question sample is small and does not measure factuality or educational usefulness.

In [15]:
eligible = comparison_df[comparison_df["successful_requests"] > 0].copy()
if eligible.empty:
    raise RuntimeError("No successful requests; cannot select a prompt for production.")

ranked = eligible.sort_values(
    by=["valid_json_rate", "avg_cost_per_request_usd", "avg_latency_seconds"],
    ascending=[False, True, True],
)
winner = ranked.iloc[0]
winner_label = winner["prompt_version"]
winner_registry_version = int(winner["mlflow_prompt_version"])
display(ranked)
print("Selected prompt:", winner_label)
print("Registry version:", winner_registry_version)

,prompt_version,mlflow_prompt_version,avg_latency_seconds,avg_input_tokens,avg_output_tokens,avg_cost_per_request_usd,projected_monthly_cost_10000_usd,valid_json_rate,successful_requests,mlflow_run_id
0,v1,1,0.093641,23.4,0,0.000001,0.0117,0.0,5,019b95b4d4cb43dba9980f1ebfd8263e
1,v2,2,0.095295,47.4,0,0.000002,0.0237,0.0,5,4f2bce36abc340f8828614eac91e8c0b
2,v3,3,0.092399,87.4,0,0.000004,0.0437,0.0,5,1333574a702741bebda00b1f5bd52f8f


Selected prompt: v1
Registry version: 1


## 12. Promote the selected version using the `production` alias

The alias is set only after evaluation. The next cell reloads the alias and verifies the resolved prompt version.

In [16]:
mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias="production",
    version=winner_registry_version,
)
production_prompt = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}@production",
    cache_ttl_seconds=0,
)
print("Production alias prompt:", production_prompt.name)
print("Production alias version:", production_prompt.version)

Production alias prompt: eduassist_academic_qa
Production alias version: 1


## 13. Save CSV outputs and latency summary

The request-level CSV contains results for all 15 calls (five questions × three prompt versions). Its v1 rows provide the Task 1 baseline measurements.

In [17]:
latency_tokens_cost_path = OUTPUT_DIR / "latency_tokens_cost.csv"
prompt_evaluation_path = OUTPUT_DIR / "prompt_evaluation_results.csv"
latency_summary_path = OUTPUT_DIR / "latency_summary.json"

results_df.to_csv(latency_tokens_cost_path, index=False)
comparison_df.to_csv(prompt_evaluation_path, index=False)
latency_summary_path.write_text(json.dumps(latency_summary, indent=2), encoding="utf-8")

print("Saved:", latency_tokens_cost_path)
print("Saved:", prompt_evaluation_path)
print("Saved:", latency_summary_path)

Saved: outputs/latency_tokens_cost.csv
Saved: outputs/prompt_evaluation_results.csv
Saved: outputs/latency_summary.json


## 14. Generate a report draft from actual results

This creates a Markdown report populated with measured metrics. Review and expand it to the required 500–700 words, including your interpretation and screenshots, before exporting to PDF.

In [18]:
successful_baseline = baseline_df[baseline_df["response_status"] == "success"]
if len(successful_baseline):
    slowest = successful_baseline.loc[successful_baseline["latency_seconds"].idxmax()]
    slowest_explanation = (
        f"The slowest baseline request was {slowest['question_id']} at "
        f"{slowest['latency_seconds']:.3f} seconds with {int(slowest['output_tokens'])} "
        "output tokens. Possible contributors include network variability, provider load, "
        "and model processing/output length; the measurement alone cannot establish the cause."
    )
else:
    slowest_explanation = "No successful baseline request was recorded; investigate the API errors."

report = f'''# EduAssist AI – LLMOps Assignment 2 Report

## Objective and setup
This experiment measures Groq API latency, token consumption, estimated request cost, and prompt-version performance for an academic question-answering assistant. The selected model was {MODEL_NAME}. The MLflow experiment was {EXPERIMENT_NAME}. The pricing date recorded was {PRICING_DATE}; the configured rates were ${INPUT_PRICE_PER_MILLION} USD per million input tokens and ${OUTPUT_PRICE_PER_MILLION} USD per million output tokens. These rates must be checked against published pricing before submission.

## Task 1 – Latency
Five academic questions were executed using baseline prompt v1. Mean latency: {latency_summary["mean"]}; median: {latency_summary["median"]}; minimum: {latency_summary["minimum"]}; maximum: {latency_summary["maximum"]}; sample standard deviation: {latency_summary["standard_deviation"]}. {slowest_explanation} Latency may vary with prompt and response length, network conditions, provider load, and model choice.

## Task 2 – Token usage and cost
Input, output, and total token counts were extracted from API usage metadata. Request cost was calculated as input-token cost plus output-token cost. The monthly projection multiplies average successful-request cost by 10,000. This estimate depends on the configured rates and the small test sample, and is not a guaranteed bill. Cost can be optimized by shortening unnecessary context, limiting response length, caching appropriate repeated content, and choosing a lower-cost model when quality is acceptable.

## Task 3 – Prompt evaluation and promotion
Prompt v1 is the basic instruction, v2 adds concise and accurate undergraduate-level academic guidance, and v3 requires strict JSON with answer, confidence, and key_points. All three versions were tested on the same five questions. The comparison metrics include average latency, input tokens, output tokens, cost per request, and JSON schema-validity rate. The selected version was {winner_label}, MLflow registry version {winner_registry_version}. Its JSON-validity rate was {winner["valid_json_rate"]:.1%}, average latency was {winner["avg_latency_seconds"]:.3f} seconds, and average request cost was ${winner["avg_cost_per_request_usd"]:.8f} USD. The production alias was assigned to this version and then reloaded for verification.

## Conclusion and limitations
The experiment demonstrates request observability, token-based cost estimation, prompt versioning, evaluation, and promotion through an MLflow alias. The selection rule prioritizes schema validity, then cost, then latency. Five questions are insufficient to establish factual accuracy or educational quality. A production rollout should add a larger representative evaluation set, human-reviewed quality and factuality checks, monitoring, and rollback procedures. Model availability and pricing should be rechecked before deployment.
'''
report_path = OUTPUT_DIR / "assignment_report_draft.md"
report_path.write_text(report, encoding="utf-8")
print("Report saved:", report_path)
print("Word count:", len(report.split()))

Report saved: outputs/assignment_report_draft.md
Word count: 396


In [23]:
!mlflow ui --backend-store-uri sqlite:///mlflow.db --host 0.0.0.0 --port 5000

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/11/01 12:57:58 INFO mlflow.server.jobs.executor_registry: Available backends after discovery: local
2026/11/01 12:57:58 INFO mlflow.server.jobs.executor_registry: Job executor backend 'local' passed requirements check
2026/11/01 12:57:58 INFO:     Uvicorn running on http://0.0.0.0:5000 (Press CTRL+C to quit)
2026/11/01 12:57:58 INFO:     Started parent process [2497]
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' curr

In [25]:
from google.colab.output import eval_js

# Retrieve the secure external proxy URL
target_port = 5000
proxy_url = eval_js(f"google.colab.kernel.proxyPort({target_port})")

print("🔗 Click the link below to access your MLflow Dashboard:")
print(proxy_url)

🔗 Click the link below to access your MLflow Dashboard:
https://5000-gpu-t4-s-usw4a0-2hzudr5hfron4-a.us-west4.prod.r.colab.dev


In [27]:
import subprocess
import time
import urllib.request

# 1. Kill any existing process bound to port 5000
print("Stopping existing processes on port 5000...")
subprocess.run("fuser -k 5000/tcp", shell=True, capture_output=True)
time.sleep(1)

# 2. Start MLflow UI as a background daemon with the --dev flag to bypass host headers
print("Starting MLflow background daemon with host-checking disabled...")
subprocess.Popen("nohup mlflow ui --backend-store-uri sqlite:///mlflow.db --host 0.0.0.0 --port 5000 --dev > mlflow.log 2>&1 &", shell=True)
time.sleep(3)

# 3. Install localtunnel globally via npm
print("Verifying localtunnel installation...")
subprocess.run("npm install -g localtunnel", shell=True, capture_output=True)

# 4. Find the public IP address of the Colab VM
endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"\n🔑 YOUR TUNNEL PASSWORD (IP): {endpoint_ip}")
print("Copy this IP. You will need to paste it as the password on the localtunnel page.\n")

# 5. Start the tunnel and print the direct web link
print("Opening public tunnel connection...")
process = subprocess.Popen("lt --port 5000", shell=True, stdout=subprocess.PIPE)

# Read and print the URL generated by localtunnel
for line in iter(process.stdout.readline, b''):
    decoded_line = line.decode('utf-8').strip()
    if "url is:" in decoded_line:
        print(f"\n🔗 MLFLOW PUBLIC URL: {decoded_line.split('url is:')[-1].strip()}")
        break

Stopping existing processes on port 5000...
Starting MLflow background daemon with host-checking disabled...
Verifying localtunnel installation...

🔑 YOUR TUNNEL PASSWORD (IP): 34.16.216.147
Copy this IP. You will need to paste it as the password on the localtunnel page.

Opening public tunnel connection...

🔗 MLFLOW PUBLIC URL: https://curly-kids-eat.loca.lt


## 15. MLflow UI and required screenshots

Start the UI in another Colab cell:

```bash
!mlflow ui --backend-store-uri sqlite:///mlflow.db --host 0.0.0.0 --port 5000
```

Use Colab's port proxy or another approved port-forwarding method to open it. Capture screenshots from your actual run showing:

1. Experiment `eduassist_llmops_assignment`, its runs, and metrics.
2. Prompt Registry with v1, v2, and v3.
3. Prompt comparison or prompt diff evidence.
4. The `production` alias and the version it resolves to.

Do not fabricate screenshots or experimental results. MLflow UI navigation may differ slightly by release.

## 16. Package outputs for download

This creates a ZIP of the result CSVs, JSON summary, report draft, and MLflow database. Download this notebook separately using **File → Download → Download .ipynb**.

In [22]:
import zipfile
from pathlib import Path

zip_path = Path("eduassist_llmops_assignment2_outputs.zip")
files_to_package = [
    OUTPUT_DIR / "latency_tokens_cost.csv",
    OUTPUT_DIR / "prompt_evaluation_results.csv",
    OUTPUT_DIR / "latency_summary.json",
    OUTPUT_DIR / "assignment_report_draft.md",
    Path("mlflow.db"),
]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for item in files_to_package:
        if item.exists():
            archive.write(item, arcname=item.name)

print("Created:", zip_path.resolve())
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download the ZIP from the notebook file browser.")

Created: /content/eduassist_llmops_assignment2_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 17. Submission checklist

- [ ] Executed Jupyter notebook (`.ipynb`)
- [ ] Exported notebook as PDF or HTML
- [ ] `requirements.txt`
- [ ] `latency_tokens_cost.csv`
- [ ] `prompt_evaluation_results.csv`
- [ ] MLflow experiment screenshot with runs and metrics
- [ ] Prompt Registry screenshot showing v1, v2, and v3
- [ ] Prompt comparison/diff evidence
- [ ] Production alias screenshot
- [ ] Final 500–700 word PDF report
- [ ] README with Python version, libraries, environment variables, installation commands, MLflow UI command, and reproduction steps

**Academic integrity and reproducibility:** Record Python, MLflow, and Groq SDK versions; model name; API/pricing date; and currency. Never include API keys, billing information, or personal data in the notebook, screenshots, or ZIP.